# Alpha model — data exploration

Inspect the shared `data_pipeline` and prototype features here before moving logic into `models/alpha/features.py`.

**Cursor / VS Code note:** Long cells often show **no output until the cell finishes** (output is buffered). The running square = still working. First `get_data()` takes ~1–2 min.

1. Run **"Kernel OK"** cell first (instant).
2. Run **load** cell and wait ~2 min without re-running.
3. Second session loads from `notebooks/.cache/` (fast).

```bash
uv sync
uv run jupyter lab notebooks/alpha_data_exploration.ipynb
```

In [1]:
# Run this first — should print instantly. If not, kernel/env is wrong.
print("Kernel OK — Python is running", flush=True)

Kernel OK — Python is running


In [2]:
import pickle
import time
from pathlib import Path

from data_pipeline import DataSplit, DataVariant, get_data

_repo_root = Path.cwd()
if (_repo_root / "models").exists():
    cache_dir = _repo_root / "notebooks" / ".cache"
else:
    cache_dir = _repo_root / ".cache"
CACHE_FILE = cache_dir / "with_fundamentals.pkl"

if CACHE_FILE.exists():
    print("Loading cached splits from", CACHE_FILE, flush=True)
    with CACHE_FILE.open("rb") as f:
        data = pickle.load(f)
    train = data[DataSplit.TRAIN]
    print("Cache load done.", flush=True)
else:
    print("No cache — calling get_data() (~1–2 min). Output may appear only when finished.", flush=True)
    start = time.perf_counter()
    data = get_data(DataVariant.WITH_FUNDAMENTALS)
    elapsed = time.perf_counter() - start
    train = data[DataSplit.TRAIN]
    cache_dir.mkdir(parents=True, exist_ok=True)
    with CACHE_FILE.open("wb") as f:
        pickle.dump(data, f)
    print(f"get_data() done in {elapsed:.1f}s — cached to {CACHE_FILE}", flush=True)

print("train shape:", train.shape, flush=True)
print("tickers:", sorted(train.columns.get_level_values("Ticker").unique()), flush=True)
print("features:", sorted(train.columns.get_level_values("Feature").unique()), flush=True)

No cache — calling get_data() (~1–2 min). Output may appear only when finished.
Preparing data...
Using pinned data snapshot.
Data prepared.
Validating data...
Data validated.
Cleaning data...
Data cleaned. Rows removed: 1.
Validating cleaned data...
Cleaned data validated.
Preparing indicator-enriched data...
Indicator-enriched data prepared.
Validating indicator-enriched data...
INFO: 3060 missing values
Indicator-enriched data validated.
Preparing fundamental-enriched data...
Using pinned fundamentals snapshot.


c:\Users\vince\Documents\repos\rl-fundamental-trading\.venv\Lib\site-packages\data_pipeline\fundamentals.py:583: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  enriched_data[(feature_name, ticker)] = feature_frame[
c:\Users\vince\Documents\repos\rl-fundamental-trading\.venv\Lib\site-packages\data_pipeline\fundamentals.py:583: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  enriched_data[(feature_name, ticker)] = feature_frame[
c:\Users\vince\Documents\repos\rl-fundamental-trading\.venv\Lib\site-packages\data_pipeline\fundamentals.p

Fundamental-enriched data prepared.
Validating fundamental-enriched data...
Fundamental-enriched data validated.
Cleaning combined enriched data...
Combined enriched data cleaned. Rows removed: 33.
Validating cleaned combined enriched data...
Cleaned combined enriched data validated.
Splitting all data variants...
All data variants split.
get_data() done in 26.0s — cached to notebooks\.cache\with_fundamentals.pkl
train shape: (1974, 320)
tickers: ['AAPL', 'ADP', 'AMAT', 'COST', 'CRM', 'CSCO', 'IBM', 'ISRG', 'JNJ', 'LLY', 'LOW', 'LRCX', 'MDLZ', 'MSFT', 'MU', 'NVDA', 'PEP', 'TGT', 'TSLA', 'TXN']
features: ['Close', 'High', 'Low', 'Open', 'Volume', 'assets', 'debt_to_equity', 'filing_lag_days', 'gross_margin', 'gross_profit', 'liabilities', 'net_income', 'operating_cashflow', 'revenue', 'roe', 'stockholders_equity']


c:\Users\vince\Documents\repos\rl-fundamental-trading\.venv\Lib\site-packages\data_pipeline\fundamentals.py:583: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  enriched_data[(feature_name, ticker)] = feature_frame[
c:\Users\vince\Documents\repos\rl-fundamental-trading\.venv\Lib\site-packages\data_pipeline\pipeline.py:100: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  combined_enriched_data[column] = fundamental_enriched_data[column]


## Compare data variants

`WITH_FUNDAMENTALS` = OHLCV + SEC fundamentals. Indicators live in `WITH_INDICATORS` (not merged by default).

In [3]:
fund_features = sorted(train.columns.get_level_values("Feature").unique())
print(f"WITH_FUNDAMENTALS: {train.shape} — {len(fund_features)} types")
print(fund_features)

# Optional — loads pipeline again (~30–90s):
# ind_train = get_data(DataVariant.WITH_INDICATORS)[DataSplit.TRAIN]
# print(sorted(ind_train.columns.get_level_values("Feature").unique()))

WITH_FUNDAMENTALS: (1974, 320) — 16 types
['Close', 'High', 'Low', 'Open', 'Volume', 'assets', 'debt_to_equity', 'filing_lag_days', 'gross_margin', 'gross_profit', 'liabilities', 'net_income', 'operating_cashflow', 'revenue', 'roe', 'stockholders_equity']


## One ticker, wide → long

One ML row = `(timestamp, ticker)`. Below: AAPL slice and stacked long table.

In [4]:
TICKER = "AAPL"

aapl_wide = train.xs(TICKER, level="Ticker", axis=1)
aapl_wide.head()

Feature,Close,High,Low,Open,Volume,filing_lag_days,revenue,net_income,operating_cashflow,gross_profit,assets,liabilities,stockholders_equity,debt_to_equity,gross_margin,roe
Datetime,,,,,,,,,,,,,,,,
2024-09-19 14:30:00-04:00,229.467194,229.490005,228.760101,228.869995,4815301,34.0,8.577700e+10,2.144800e+10,2.885800e+10,3.967800e+10,3.316120e+11,2.649040e+11,6.670800e+10,3.971098,0.462572,0.321521
2024-09-19 15:30:00-04:00,228.820007,229.589996,228.289993,229.460007,7376731,34.0,8.577700e+10,2.144800e+10,2.885800e+10,3.967800e+10,3.316120e+11,2.649040e+11,6.670800e+10,3.971098,0.462572,0.321521
2024-09-20 09:30:00-04:00,230.034500,231.160004,229.220001,229.970001,28984531,34.0,8.577700e+10,2.144800e+10,2.885800e+10,3.967800e+10,3.316120e+11,2.649040e+11,6.670800e+10,3.971098,0.462572,0.321521
2024-09-20 10:30:00-04:00,229.770004,230.580002,229.199997,230.029999,12497143,34.0,8.577700e+10,2.144800e+10,2.885800e+10,3.967800e+10,3.316120e+11,2.649040e+11,6.670800e+10,3.971098,0.462572,0.321521
2024-09-20 11:30:00-04:00,231.729996,231.740005,229.740005,229.759995,7914719,34.0,8.577700e+10,2.144800e+10,2.885800e+10,3.967800e+10,3.316120e+11,2.649040e+11,6.670800e+10,3.971098,0.462572,0.321521


In [9]:
long = train.stack("Ticker", future_stack=True)
long.index.names = ["Datetime", "Ticker"]
long.head(35)

Feature                                Close        High         Low  \
Datetime                  Ticker                                       
2024-09-19 14:30:00-04:00 AAPL    229.467194  229.490005  228.760101   
                          ADP     277.100006  278.024994  277.100006   
                          AMAT    196.354996  197.774994  195.720001   
                          COST    901.994873  903.719971  899.840088   
                          CRM     265.214996  266.489990  264.799988   
                          CSCO     51.560001   51.630001   51.494999   
                          IBM     213.645004  213.679993  213.070007   
                          ISRG    487.609985  488.480011  487.329987   
                          JNJ     165.070007  165.679993  165.070007   
                          LLY     913.809998  915.080017  912.500000   
                          LOW     260.725006  261.600006  260.540009   
                          LRCX     78.949997   79.787498   78.784996   
                          MDLZ     74.989998   75.114998   74.820000   
                          MSFT    438.980011  439.600006  438.440094   
                          MU       89.349998   90.025002   89.129997   
                          NVDA    118.144997  119.135002  117.610001   
                          PEP     174.710007  175.054993  174.500000   
                          TGT     156.970001  156.979996  156.419998   
                          TSLA    242.839996  243.990005  241.750000   
                          TXN     207.279999  209.360001  207.100006   
2024-09-19 15:30:00-04:00 AAPL    228.820007  229.589996  228.289993   
                          ADP     277.690002  277.850006  276.920105   
                          AMAT    196.699997  198.149994  196.111496   
                          COST    901.000000  903.599976  899.159973   
                          CRM     265.989990  266.470001  264.970001   
                          CSCO     51.439999   51.564999   51.250000   
                          IBM     213.889999  214.009995  213.294998   
                          ISRG    490.010010  490.149994  486.989990   
                          JNJ     164.869995  165.050003  164.619995   
                          LLY     915.619995  916.809998  913.099976   
                          LOW     261.000000  261.019989  260.130005   
                          LRCX     78.921997   79.529495   78.771004   
                          MDLZ     74.779999   74.985001   74.739998   
                          MSFT    438.679993  439.320007  437.709991   
                          MU       89.230003   89.827202   89.044998   

Feature                                 Open      Volume  filing_lag_days  \
Datetime                  Ticker                                            
2024-09-19 14:30:00-04:00 AAPL    228.869995   4815301.0             34.0   
                          ADP     277.839996     99147.0             38.0   
                          AMAT    197.774994    591063.0             25.0   
                          COST    900.080017    143042.0             25.0   
                          CRM     266.470001    886799.0             29.0   
                          CSCO     51.570000   1877784.0             40.0   
                          IBM     213.149994    409296.0             30.0   
                          ISRG    488.359985     85279.0             19.0   
                          JNJ     165.250000    403378.0             25.0   
                          LLY     914.950012    180746.0             39.0   
                          LOW     261.549988    229586.0             27.0   
                          LRCX     79.787498    156179.0             60.0   
                          MDLZ     74.820000    283922.0             30.0   
                          MSFT    438.929993   1358448.0             30.0   
                          MU       90.019997   2175015.0             28.0   
                          

## Prototype: returns, momentum, target

- Features use only information at time `t`.
- Target = forward return over `H` **trading days** (not hours).
- Data is hourly (~7 bars/day) → `H_bars = horizon_days * bars_per_day`.

In [10]:
import pandas as pd

HORIZON_DAYS = 5
BARS_PER_DAY = 7
H_BARS = HORIZON_DAYS * BARS_PER_DAY

close = train["Close"]
return_1bar = close / close.shift(1) - 1
momentum_20bar = close / close.shift(20) - 1

fwd_return = close.shift(-H_BARS) / close - 1

# AAPL example series
pd.DataFrame({
    "close": close[TICKER],
    "return_1bar": return_1bar[TICKER],
    "momentum_20bar": momentum_20bar[TICKER],
    "target_return": fwd_return[TICKER],
}).dropna().head(10)

,close,return_1bar,momentum_20bar,target_return
Datetime,,,,
2024-09-24 13:30:00-04:00,226.879105,-0.001588,-0.011279,-0.006255
2024-09-24 14:30:00-04:00,227.026001,0.000647,-0.007840,-0.003858
2024-09-24 15:30:00-04:00,227.360001,0.001471,-0.011627,-0.004838
2024-09-25 09:30:00-04:00,226.384995,-0.004288,-0.014732,-0.004263
2024-09-25 10:30:00-04:00,225.449997,-0.004130,-0.027101,0.003349
2024-09-25 11:30:00-04:00,225.870102,0.001863,-0.021148,0.004790
2024-09-25 12:30:00-04:00,225.360001,-0.002258,-0.019224,0.007099
2024-09-25 13:30:00-04:00,224.899994,-0.002041,-0.030499,0.008537
2024-09-25 14:30:00-04:00,225.259903,0.001600,-0.014309,0.008790


## Workflow tip

1. Prototype new columns here (play with horizons, normalization, indicator merge).
2. When happy, copy the logic into `models/alpha/features.py` → `build_dataset()`.
3. Keep the notebook for exploration; don't import notebook code from production paths.